# L10b Lab: Implementation of the Weighted Majority Algorithm for Online Learning

In this lab, we will implement and explore the weighted majority algorithm for online learning in a daily stock market guessing game.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Implement the weighted majority algorithm:__ The weighted majority algorithm combines the predictions of several experts with a weighted vote and lowers the weight of each expert that makes a mistake. We'll complete the vote and the weight update ourselves and check our implementation against the course package.
> * __Track the expert weights:__ The weight of each expert records how many mistakes it has made so far. We'll plot the weights over a year of trading days and see which expert the algorithm learns to trust.
> * __Test the mistake bound:__ The weighted majority theorem limits our number of mistakes by a multiple of the best expert's mistakes plus a constant. We'll compare both sides of the bound during the game, and again when the experts must forecast the next day.

In [the L10a lecture](../L10a/CHEME-5800-L10a-Lecture-OnlineLearningAndRegret-Fall-2026.ipynb), we introduced online learning from expert advice and the weighted majority algorithm. Here, five experts advise us each day on whether the SPY exchange-traded fund went up or down, and the algorithm learns from their mistakes which experts to trust.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

Your weighted majority algorithm goes in [src/Compute.jl](src/Compute.jl). We call [the `L10bWMA.my_weighted_majority(...)` function](src/Compute.jl) by its qualified name, so that re-running the setup cell picks up your edits.

### Data
We gathered a daily open-high-low-close dataset for each firm in the S&P 500 from `01-03-2014` until `12-31-2024`, along with data for a few exchange-traded funds during that time. An __exchange-traded fund__ (ETF) holds a basket of stocks and trades like a single stock; the SPY ETF tracks the S&P 500 index.

This lab's copy of the dataset, in the [data](data) folder, keeps the 424 tickers with a record for every one of the 2,767 trading days. For each ticker, it keeps the daily __volume-weighted average price__, the average price of the day's trades weighted by the number of shares traded. The [data README](data/README.md) describes how the copy was made.

Let's load the data with [the `CSV.read(...)` function](https://csv.juliadata.org/stable/reading.html#CSV.read) and store one `DataFrame` per ticker in the `dataset::Dict{String,DataFrame}` dictionary, with the ticker symbol as the key:

In [ ]:
dataset = let

    # initialize -
    path = joinpath(CHEME5800_L10B_DATA, "SP500-Daily-VWAP-1-3-2014-to-12-31-2024.csv"); # the market data cache
    prices = CSV.read(path, DataFrame); # one row per trading day: the date, then one column per ticker
    dataset = Dict{String,DataFrame}(); # ticker => that ticker's prices

    # build one DataFrame per ticker -
    for ticker ∈ names(prices)[2:end] # column 1 is the date, so skip it
        dataset[ticker] = DataFrame(timestamp = prices[:, :date],
            volume_weighted_average_price = prices[:, ticker]);
    end

    # return -
    dataset
end;

Next, let's get a list of the firms in our dataset (sorted alphabetically). We store the sorted firm ticker symbols in the `list_of_tickers::Array{String,1}` variable:

In [ ]:
list_of_tickers = keys(dataset) |> collect |> sort # list of firm "ticker" symbols in alphabetical order

Finally, let's set up a ticker map that holds the index of each ticker value. We'll save this in the `tickerindexmap::Dict{String,Int}` dictionary:

In [ ]:
tickerindexmap = let

    # initialize -
    tickerindexmap = Dict{String,Int}();
    for i ∈ eachindex(list_of_tickers)
        tickerindexmap[list_of_tickers[i]] = i;
    end

    tickerindexmap;
end

___

## Task 1: Set Up the Expert Advice Game
In this task, we compute the daily growth rates and write the functions that play the experts and the adversary. The game is the _prediction from expert advice_ problem from the L10a lecture:

* __Game__: The game's objective is to predict the daily movement of the SPY ETF as either `up` or `down` (which we encode as `{1 | -1}`). If we predict _incorrectly_, we lose a dollar. We aim to minimize losses. Our predictions are informed by the predictions of $n$ experts, which may be correlated and accurate or inaccurate.
* __Goal__: The weighted majority algorithm limits losses to about the best expert without knowing who that is until the end of the sequence. Each day requires a decision. The algorithm maintains weights for experts, initially giving them equal weight. Over time, experts who make mistakes have their weights decreased, while those who predict correctly maintain higher relative influence.

We play this game between an omniscient _adversary_ (nature, i.e., the market) and an _aggregator_ (us) who $n$ experts advise; if the weighted vote is tied, we predict `-1`. The game proceeds in rounds $t = 1, 2, \ldots, T$.

### Compute the growth rate matrix
Next, let's compute the growth rate array, which contains, for each day and each firm in our dataset, the value of the growth rate between time $j$ and $j-1$.

>  __Continuously Compounded Growth Rate (CCGR)__
>
> Let's assume a model of the share price of firm $i$ is governed by an expression of the form:
>
>$$
\begin{align*}
S^{(i)}_{j} &= S^{(i)}_{j-1}\;\exp\left(g^{(i)}_{j,j-1}\Delta{t}_{j}\right)
\end{align*}
$$
>
> where $S^{(i)}_{j-1}$ denotes the share price of firm $i$ at time index $j-1$, $S^{(i)}_{j}$ denotes the share price of firm $i$ at time index $j$, and $\Delta{t}_{j} = t_{j} - t_{j-1}$ denotes the length of a time step (units: years) between time index $j-1$ and $j$. The value we are going to estimate is the growth rate $g^{(i)}_{j,j-1}$ (units: inverse years) for each firm $i$, and each time step in the dataset.

We've implemented [the `log_growth_matrix(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.log_growth_matrix), which takes the dataset and a list of ticker symbols, and returns the growth rate array. Each row of the growth rate array is a time step, while each column corresponds to a firm from the `list_of_tickers::Array{String,1}` array. Row $t$ holds the growth from trading day $t$ to trading day $t+1$, so round $t$ of our game is trading day $t+1$.

In [ ]:
growth_rate_array = let

    # initialize -
    τ = (1/252); # time-step one-day in units of years (trading year is 252 days)
    r̄ = 0.0; # assume the risk-free rate is 0

    # compute the growth matrix -
    growth_rate_array = log_growth_matrix(dataset, list_of_tickers, Δt = τ,
        risk_free_rate = r̄); # other optional parameters are at their defaults

    growth_rate_array; # return
end;

### Experts and adversary
Each expert is another ticker, and it predicts that SPY moved in the same direction as its own ticker that day. We use the QQQ ETF (which tracks the 100 largest non-financial firms listed on the Nasdaq exchange), NVIDIA (`NVDA`), AMD (`AMD`), Apple (`AAPL`), and Microsoft (`MSFT`). Each expert reads the same day's growth rate as SPY's, which we only know once the market closes, so this game tests how well a weighted vote combines correlated signals, not how well it forecasts. In Task 3, we make the experts forecast.

> __Expert function__: The `expert(index::Int, time::Int, data::Array{Float64,2})::Int` function takes the `index::Int` of the expert, the round `time::Int`, and the `data::Array{Float64,2}` array (the growth rate array). It returns the _prediction_ of expert $i$ in round $t$, namely $\hat{y}_{i}^{(t)}$: the sign of its ticker's growth rate, which is `1` (up), `-1` (down), or `0` if the price did not change.

Let's take a look at the implementation of the `expert(...)` function:

In [ ]:
function expert(index::Int, time::Int, data::Array{Float64,2})::Int

    # These are my choices for experts: you pick yours
    # These experts use the growth rate data from other tickers to predict the SPY direction
    # But we don't know which ones are best!
    # expert 1: QQQ
    # expert 2: NVDA
    # expert 3: AMD
    # expert 4: AAPL
    # expert 5: MSFT

    i = nothing;
    if (index == 1)
        i = tickerindexmap["QQQ"]; # expert one uses QQQ growth rate data: SPY and QQQ are highly correlated
    elseif (index == 2)
        i = tickerindexmap["NVDA"];
    elseif (index == 3)
        i = tickerindexmap["AMD"];
    elseif (index == 4)
        i = tickerindexmap["AAPL"];
    elseif (index == 5)
        i = tickerindexmap["MSFT"];
    end

    return data[time,i] |> sign; # 1 (up), -1 (down), or 0 (no change)
end;

Next, we'll define the `adversary(...)` function.

> __Adversary function__: The `adversary(time::Int, data::Array{Float64,2})::Int` function takes the round `time::Int` and the `data::Array{Float64,2}` array and returns the true answer $y_{t}$, the direction SPY moved in round $t$.

Take a look at the implementation of the `adversary(...)` function:

In [ ]:
function adversary(time::Int, data::Array{Float64,2})::Int

    # Adversary: This is the true answer for the direction of SPY
    # The adversary uses the actual SPY growth rate data
    result = tickerindexmap["SPY"] |> i-> data[time,i] |> sign;
    return result
end;

Let's look at the first ten rounds of the game, the predictions of the five experts and the direction SPY actually moved:

In [ ]:
let

    # initialize -
    n = 5; # number of experts
    number_of_rounds = 10; # how many rounds to show
    df = DataFrame(round = 1:number_of_rounds); # one row per round

    # each expert's predictions, one column per expert -
    for i ∈ 1:n
        df[!, "expert $(i)"] = [expert(i, t, growth_rate_array) for t ∈ 1:number_of_rounds];
    end
    df[!, "actual (SPY)"] = [adversary(t, growth_rate_array) for t ∈ 1:number_of_rounds]; # the direction SPY moved

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1) # show every row and column
    );
end

__What do we see?__ On some days, all five experts agree with SPY. On others, the experts split, and in round 4 all five predicted down while SPY went up.

### Things to think about

__Question:__ Each expert reads its ticker's growth rate for the same day as SPY's. Could we use this advice to decide whether to buy SPY that morning? What would an expert have to use instead? Briefly explain.

___

## Task 2: Implement the Weighted Majority Algorithm
In this task, we implement the weighted majority algorithm and play the game for $T = 252$ rounds, about one year of trading days. The algorithm from the L10a lecture is:

__Initialize__: Fix a learning rate $0<\epsilon\leq{1/2}$, and give each expert the weight $w_{i}^{(1)} = 1$ for $i = 1, 2, \ldots, n$.

For each round $t=1,2,\dots,T$ __do__:
1. The aggregator (us) makes a prediction $\hat{y}_t \in \{-1, 1\}$ based on the weighted majority of the experts' predictions. If the total weight of the experts predicting `1` is greater than the total weight of the experts predicting `-1`, we predict `1`; otherwise, we predict `-1`.
2. The adversary (market) reveals the actual outcome $y_t \in \{-1, 1\}$.
3. We decrease the weights of the experts who predicted incorrectly. For each expert $i$ who predicted incorrectly, we update the weight: $w_{i}^{(t+1)} = w_{i}^{(t)}(1-\epsilon)$. The other weights do not change, $w_{i}^{(t+1)} = w_{i}^{(t)}$.

An expert that predicts `0` (no change) counts toward neither side of the vote, and it counts as a mistake.

### Implement the Weighted Majority Algorithm
Open [`src/Compute.jl`](src/Compute.jl) and complete [the `my_weighted_majority(...)` function](src/Compute.jl). Everything except three lines is already written, and a TODO comment marks each of them:

> __Implementation tasks__
>
> * __TODO 1:__ Our prediction (step 1). Replace the right-hand side of `ŷ = 0` with `1` when the experts predicting up carry more weight than those predicting down, and `-1` otherwise.
> * __TODO 2:__ Each expert's mistake (step 3). Replace the right-hand side of `mistake = 0` with `1` when expert $i$ predicted incorrectly, and `0` otherwise.
> * __TODO 3:__ The weight update (step 3). Replace the right-hand side of `weights[t+1, i] = weights[t, i]` with the weight multiplied by $1-\epsilon\cdot\texttt{mistake}$.

Save the file and re-run the setup cell at the top of the notebook to reload your edits, then run the cells below. Until TODO 1 is done, the call throws an error.

Next, we'll build [a `model::MyBinaryWeightedMajorityAlgorithmModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/online/#VLDataScienceMachineLearningPackage.MyBinaryWeightedMajorityAlgorithmModel) that holds all the data associated with the problem, using [the `build(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/). We use the learning rate $\epsilon = 10/T$:

In [ ]:
model = let

    # initialize -
    T = 252; # number of rounds (days)
    ϵ = 10.0/T; # learning rate (you can tune this parameter)

    # build -
    model = build(MyBinaryWeightedMajorityAlgorithmModel, (
        ϵ = ϵ, # learning rate
        n = 5, # number of experts
        T = T, # number of rounds
        expert = expert, # expert function
        adversary = adversary, # adversary function
    ));

    # return the model -
    model;
end;

Now, let's play the game with [the `L10bWMA.my_weighted_majority(...)` function](src/Compute.jl). It takes the `model::MyBinaryWeightedMajorityAlgorithmModel` instance and the growth rate array and returns two arrays:
* The `sims::Array{Int64,2}` array holds the record of the game. Each row corresponds to a round `t`. The first $n$ columns hold the predictions of experts $i=1,2,\dots,n$ in round `t`. Column $n+1$ holds the aggregator (us) prediction, and column $n+2$ holds the actual outcome revealed by the adversary. The last column holds our mistake: `1` if our prediction was incorrect, otherwise `0`.
* The `weights::Array{Float64,2}` array holds the weights of the experts. Row `t` holds the weights $w_{i}^{(t)}$ used in round `t`, so the first row is all ones and the last row, row $T+1$, holds the weights after the last round. Each column corresponds to an expert.

In [ ]:
(sims, weights) = L10bWMA.my_weighted_majority(model, growth_rate_array); # the untouched stub throws an error

### Check: Your Weighted Majority Algorithm

The course package plays the same game with [the `play(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/online/#VLDataScienceMachineLearningPackage.play). Let's check that your function agrees with it in every round, for our learning rate and for a much larger one, $\epsilon = 1/2$:

In [ ]:
@testset "My weighted majority algorithm" begin

    # the record has one row per round, and the weights one more row than that -
    @test size(sims) == (model.T, model.n + 3)
    @test size(weights) == (model.T + 1, model.n)

    # the same game, played by the course package -
    (sims_package, weights_package) = play(model, growth_rate_array);
    @test sims == sims_package
    @test weights ≈ weights_package

    # a second game with a much larger learning rate -
    fast_model = build(MyBinaryWeightedMajorityAlgorithmModel, (
        ϵ = 0.5, n = model.n, T = model.T, expert = model.expert, adversary = model.adversary));
    (sims_fast, weights_fast) = L10bWMA.my_weighted_majority(fast_model, growth_rate_array);
    (sims_fast_package, weights_fast_package) = play(fast_model, growth_rate_array);
    @test sims_fast == sims_fast_package
    @test weights_fast ≈ weights_fast_package
end;

### Visualize the expert weights
The code below plots the weight of each expert against the round $t$. Expert 1 is QQQ, expert 2 is NVDA, expert 3 is AMD, expert 4 is AAPL, and expert 5 is MSFT.

__Prediction:__ All experts start with equal weights. Which expert do you think will end the year with the largest weight?

In [ ]:
let

    # initialize -
    T = model.T; # number of rounds
    n = model.n; # number of experts
    p = plot(framestyle = :box, legend = :topright); # empty plot, then one line per expert

    # plot the weight of each expert -
    for i ∈ 1:n
        plot!(p, 1:(T+1), weights[:, i], lw = 2, label = "expert $(i)");
    end
    xlabel!(p, "Round t (trading day)");
    ylabel!(p, "Weight (AU)");
    p
end

__What do we see?__ From about round 22 on, QQQ (expert 1) has the largest weight, and at the end of the year its weight is more than four times that of any other expert.

What fraction of the time did the aggregator (us) predict correctly? Let's count the mistakes of each expert and our own:

In [ ]:
let

    # initialize -
    n = model.n; # number of experts
    T = model.T; # number of rounds
    actual = sims[:, n+2]; # the actual outcomes are in column n+2
    df = DataFrame(); # one row per player

    # each expert's mistakes -
    for i ∈ 1:n
        mistakes = count(sims[:, i] .!= actual); # rounds where expert i was incorrect
        push!(df, (player = "expert $(i)", mistakes = mistakes, fraction_correct = 1 - mistakes/T));
    end

    # ... and ours -
    mistakes = sum(sims[:, end]); # our mistakes are in the last column
    push!(df, (player = "aggregator (us)", mistakes = mistakes, fraction_correct = 1 - mistakes/T));

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = [:l, :r, :r] # left-align text and right-align numerical columns
    );
end

__What do we see?__ We predicted correctly in 208 of the 252 rounds (82.5%), with 44 mistakes. QQQ, the expert with the largest weight, made 40 mistakes, and every other expert made at least 76. If you change the experts or the learning rate, your numbers will differ.

### Things to think about

__Question:__ Our weighted vote made more mistakes than QQQ, even though QQQ had the largest weight for most of the year. How can the vote do worse than its best expert? What do you expect to happen to our mistakes if we use the larger learning rate $\epsilon = 1/2$? Briefly explain.

___

## Task 3: Test the Theoretical Mistake Bound
In this task, we'll test the theorem governing the upper bound on the number of mistakes an aggregator (us) will make relative to the system's experts.

> __Theorem__: The weighted majority algorithm has the following theoretical guarantee (which bounds the number of mistakes the aggregator makes). Let $M_{i}^{(t)}$ be the number of mistakes made by expert $i$ up to time $t$ and $M^{(t)}$ be the total number of mistakes made by the aggregator (us) up to time $t$. Then, for every expert $i$ and the aggregator, we have:
>
>$$
\begin{align*}
M^{(t)} \leq \frac{2\ln(n)}{\epsilon} + 2\left(1+\epsilon\right)M_{i}^{(t)}
\end{align*}
$$

Let's start by identifying the best expert in hindsight.

__Which expert was best?__ Each mistake multiplies an expert's weight by $1-\epsilon$, so after $T$ rounds the weight of expert $i$ is $(1-\epsilon)^{M_{i}^{(T)}}$. The expert with the largest final weight is the expert with the fewest mistakes, $\hat{i} = \arg\max_{i}\,w_{i}^{(T+1)}$:

In [ ]:
î = argmax(weights[end,:]) # index of the largest weight at the end

Next, we'll compute the number of mistakes by our experts as a function of time $t$. We'll save this in the `m_experts::Array{Int64,2}` array, where each row corresponds to a time index $t$, and each column is the loss suffered by each expert.

In this case, if the expert was _incorrect_, they have a loss of `1`; otherwise, they have a loss of `0`.

In [ ]:
m_experts = let

    # data -
    n = model.n; # number of experts
    T = model.T; # number of rounds in the game
    m = Array{Int64,2}(undef, T, n);

    for t ∈ 1:T
        for j ∈ 1:n
            expert_prediction = sims[t,j];
            adversary_prediction = sims[t,end-1]; # adv is the penultimate column
            m[t,j] = expert_prediction == adversary_prediction ? 0 : 1
        end
    end

    # return
    m
end;

### Compute the bound
Finally, let's compute the bound. We save the left side of the bound (the aggregator term) in the `L::Array{Float64,1}` variable, while the right side is saved in the `R::Array{Float64,1}` variable.

In [ ]:
L,R = let

    # data -
    n = model.n; # number of experts
    T = model.T; # number of rounds in the game
    ϵ = model.ϵ; # Get the learning rate
    L = Array{Float64,1}(undef, T); # left term
    R = Array{Float64,1}(undef, T); # right term
    m_aggregator = sims[:,end]; # aggregator mistakes on the last column
    m_expert = m_experts[:,î]; # best expert

    # compute the left term L
    for t ∈ 1:T
        L[t] = sum(m_aggregator[1:t]);
    end

    # compute the right term R
    for t ∈ 1:T
        R[t] = (2*log(n)/ϵ) + 2*(1+ϵ)*sum(m_expert[1:t])
    end

    L,R
end;

The code below plots the `L` and `R` components of the theoretical bound expression.

> __What do we expect?__ The left side of the inequality (the actual mistakes made by the aggregator) should always be less than or equal to the right side of the inequality (the theoretical bound).

In [ ]:
let
    p = plot(framestyle = :box, legend = :topleft); # empty plot, then the two sides of the bound
    plot!(p, L, lw = 2, label = "Left (aggregator)");
    plot!(p, R, lw = 2, label = "Right (best expert)");
    xlabel!(p, "Round t (trading day)");
    ylabel!(p, "Cumulative mistakes bound (AU)");
    p
end

__What do we see?__ The bound holds in every round. The right side starts at $2\ln(n)/\epsilon \approx 81$ before any expert has made a mistake, so after 252 rounds our 44 mistakes sit well below the bound of about 164.

### Make the experts forecast
Let's make the experts forecast: the experts still report their tickers' moves on trading day $t$, but the adversary now reveals SPY's move on the _next_ trading day. Everything else stays the same.

__Prediction:__ Will our fraction of correct predictions {increase, decrease, or stay the same}? Will the mistake bound still hold?

In [ ]:
next_day_adversary(time::Int, data::Array{Float64,2})::Int = adversary(time + 1, data); # SPY's move on the next trading day

(forecast_model, forecast_sims, forecast_weights) = let

    # build the same game, with the next-day adversary -
    forecast_model = build(MyBinaryWeightedMajorityAlgorithmModel, (
        ϵ = model.ϵ, # same learning rate
        n = model.n, # same number of experts
        T = model.T, # same number of rounds
        expert = expert, # same experts
        adversary = next_day_adversary, # the adversary reveals SPY's move on the next trading day
    ));

    # play -
    (forecast_sims, forecast_weights) = L10bWMA.my_weighted_majority(forecast_model, growth_rate_array);

    # return -
    (forecast_model, forecast_sims, forecast_weights)
end;

Let's compare our mistakes with those of the best expert in hindsight, and with a simple rule that always predicts up. We also check the bound in every round:

In [ ]:
let

    # initialize -
    n = forecast_model.n; # number of experts
    T = forecast_model.T; # number of rounds
    ϵ = forecast_model.ϵ; # learning rate
    actual = forecast_sims[:, n+2]; # SPY's moves on the next trading day

    # cumulative mistakes: ours and the best expert's -
    î = argmax(forecast_weights[end, :]); # the best expert in hindsight: the largest final weight
    m_ours = cumsum(forecast_sims[:, end]); # our mistakes up to each round
    m_best = cumsum(forecast_sims[:, î] .!= actual); # the best expert's mistakes up to each round
    bound = (2*log(n)/ϵ) .+ 2*(1+ϵ) .* m_best; # the right side of the bound in each round
    m_up = count(actual .!= 1); # always predicting up is wrong on every day SPY did not go up

    # show -
    println("our mistakes: ", m_ours[end], " (", round(100*(1 - m_ours[end]/T), digits=1), "% correct)");
    println("best expert (expert ", î, "): ", m_best[end], " mistakes (", round(100*(1 - m_best[end]/T), digits=1), "% correct)");
    println("always predicting up: ", m_up, " mistakes (", round(100*(1 - m_up/T), digits=1), "% correct)");
    println("bound holds in every round: ", all(m_ours .≤ bound));
end

__What do we see?__ When the experts have to forecast, our accuracy falls from 82.5% to 58.7%, with 104 mistakes. The best expert in hindsight is now AAPL (expert 4), with 101 mistakes, and always predicting up makes 106. The bound still holds in every round.

### Things to think about

__Question:__ With the next-day adversary, how does our accuracy compare with always predicting up? The mistake bound still holds. Does that mean the weighted majority algorithm forecasts SPY well? Briefly explain.

___

## Summary
In this lab, we implemented the weighted majority algorithm, used it to combine the advice of five experts on the daily movement of SPY, and tested its theoretical mistake bound.

> __Key Takeaways:__
>
> * __Weighted vote of experts:__ We completed the weighted majority algorithm, which takes a weighted vote of the experts and lowers the weight of each expert that makes a mistake. Our implementation matched the course package in every round, for a small and a large learning rate.
>
> * __Weights single out the best expert:__ Because every mistake multiplies an expert's weight by the same factor, the expert with the fewest mistakes ends with the largest weight. In the same-day game, QQQ took the lead within a month, and our weighted vote made only a few more mistakes than QQQ did.
>
> * __Mistake bound validation:__ Our mistakes stayed below the theoretical bound in every round, in the same-day game and when the experts had to forecast. The bound compares us with the best expert, so when every expert forecasts poorly, it still holds while our accuracy falls close to always predicting up.

We can now change the experts, the learning rate, or the adversary and see how the weights and our mistakes respond.

___